# Sales Analysis

This notebook runs inside a pybox sandbox. It demonstrates normal data-processing
workflows and shows what happens at the sandbox boundary.

In [ ]:
import os
import pathlib
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend, works headlessly
import matplotlib.pyplot as plt

print('pandas:', pd.__version__)
print('working directory:', os.getcwd())

## Load and inspect the data

In [ ]:
df = pd.read_csv('data/sales.csv', parse_dates=['date'])
df.head()

## Summarise by region and product

In [ ]:
summary = (
    df.groupby(['region', 'product'])
      .agg(total_units=('units', 'sum'), total_revenue=('revenue', 'sum'))
      .reset_index()
      .sort_values('total_revenue', ascending=False)
)
summary

## Plot revenue by region

In [ ]:
region_revenue = df.groupby('region')['revenue'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
region_revenue.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Total Revenue by Region')
ax.set_xlabel('Region')
ax.set_ylabel('Revenue ($)')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## Write results to the working directory

The sandbox grants full read/write access to the current working directory.
Writing files here works normally.

In [ ]:
out = pathlib.Path('output')
out.mkdir(exist_ok=True)

summary.to_csv(out / 'summary.csv', index=False)
fig.savefig(out / 'revenue_by_region.png', dpi=150)

print('Written:')
for f in sorted(out.iterdir()):
    print(' ', f)

## Sandbox boundary: attempt to write outside the working directory

The sandbox blocks writes to any path outside the current working directory.
The OS raises `PermissionError` — a clean, auditable failure.

In [ ]:
try:
    with open('/tmp/escaped.txt', 'w') as f:
        f.write('this should not be written')
    print('ERROR: write succeeded — sandbox did not block it')
except PermissionError as e:
    print(f'Blocked as expected: {e}')